In [2]:
import os
import re
import json
import pandas as pd
import time
import random
import mock_llm
from mock_llm import TransientLLMError
from mock_llm import est_tokens
from collections import Counter

## Task 1 - Input guardrails:
(a) status changed to 'skipped_empty' when the text is a empty or only whitespace.
(b) status changed to 'flagged_long' to the tickets which have more than 400 tokens.
## Task 2 - Redact Personal Information:
Using all the regular expressions and placeholders all the tickets are redacted befor it goes into prompt

In [3]:
df = pd.read_csv("data/raw/tickets.csv", keep_default_na=False)

email_count = 0
phone_count = 0
order_count = 0
redacted_texts = []

for text in df["text"]:
    text = str(text)

    text, n = re.subn(r"[\w.+-]+@[\w-]+\.[\w.]+", "[EMAIL]", text)
    email_count += n

    text, n = re.subn(r"\b[6-9]\d{9}\b", "[PHONE]", text)
    phone_count += n

    text, n = re.subn(r"ORD-\d{6}", "[ORDER_ID]", text)
    order_count += n

    redacted_texts.append(text)

results = []
messages_list = []
MAX_INPUT_TOKENS = 400

for i, text in enumerate(redacted_texts):
    original = str(df["text"][i])

    if original.strip() == "":
        status = "skipped_empty"
        messages_list.append(None)

    elif est_tokens(original) > MAX_INPUT_TOKENS:
        status = "flagged_long"
        messages_list.append(None)

    else:
        status = "pending"
        messages_list.append([{"role": "user", "content": text}])

    # runs for EVERY ticket, outside the if/elif/else
    results.append({
        "ticket_id": df["ticket_id"][i],
        "status": status,
        "est_tokens": est_tokens(original) if original.strip() else 0,
    })

# check once, after the loop
for j, m in enumerate(messages_list):
    if m is not None:
        assert m[0]["content"] == redacted_texts[j]
        assert len(m) == 1

out = pd.DataFrame(results)

print("Task 1 : Status Counts:")
print(out["status"].value_counts())
print(out[out["status"] == "flagged_long"]["est_tokens"].tolist())
print("-----------------------------------------------------------------")
print("Task 2 : Redaction list:")
print("Number of Substitutions for EMAILS :", email_count)   
print("Number of Substitutions for PHONES :", phone_count) 
print("Number of Substitutions for ORDER IDs :", order_count)
print("Total number of substitutions :", email_count + phone_count + order_count)

# check: re-apply the three patterns to the redacted texts
left = 0
for text in redacted_texts:
    left += len(re.findall(r"[\w.+-]+@[\w-]+\.[\w.]+", text))
    left += len(re.findall(r"\b[6-9]\d{9}\b", text))
    left += len(re.findall(r"ORD-\d{6}", text))

print("left after redaction:", left)

Task 1 : Status Counts:
status
pending          156
skipped_empty      4
flagged_long       4
Name: count, dtype: int64
[734, 734, 734, 734]
-----------------------------------------------------------------
Task 2 : Redaction list:
Number of Substitutions for EMAILS : 55
Number of Substitutions for PHONES : 91
Number of Substitutions for ORDER IDs : 132
Total number of substitutions : 278
left after redaction: 0


### Observation:
Observed 4 empty tickets and 4 long tickets with 734 tokens each. With this the guardrail checks stay at the top so flagged and empty tickets never reach redaction or the model.

Redaction is done for all 164 tikets. in that we have 4(Empty tickets) which have nothing to redact and 4(long tickets) which will also be redacted. And input guardrail will filer the 4(Empty tickets) and 4(long tickets) before sending it to the prompt.

## Task 3 : Prompt
Both the prompt will have CLASSIFY part and JSON response format as same. 
### Zero-short Prompt:
This type of prompt will "not have the example" tickets with the answer guidance like format, exact key-word will be mentioned explicitly. Token Usage is Less. Response may not accurate as Few-Short prompt.
### Few short Prompt:
This type of prompt will 'have the example' tickets with the answer guidance like format, exact key-word will be mentioned explicitly. Token usage is more commpared with Zero-short prompt. Response will be more accurate compared with Zer-short.

In [4]:
CATEGORIES = ["billing", "delivery", "technical", "fraud"]

# ---------- Zero-shot system prompt ----------
ZERO_SHOT_PROMPT = """You are a support ticket triage assistant.
CLASSIFY the customer ticket into exactly one of these categories:
- billing
- delivery
- technical
- fraud

Reply with JSON only, in exactly this form and nothing else:
{"category": "...", "reason": "..."}

The "category" value must be one of: billing, delivery, technical, fraud.
The "reason" value must be one short sentence.
Do not add any text, markdown or code fences outside the JSON."""

# ---------- Few-shot system prompt ----------
FEW_SHOT_PROMPT = """You are a support ticket triage assistant.
CLASSIFY the customer ticket into exactly one of these categories:
- billing
- delivery
- technical
- fraud

Reply with JSON only, in exactly this form and nothing else:
{"category": "...", "reason": "..."}

The "category" value must be one of: billing, delivery, technical, fraud.
The "reason" value must be one short sentence.
Do not add any text, markdown or code fences outside the JSON.

EXAMPLES:
Ticket: I was charged twice for my order [ORDER_ID], please refund the extra amount.
Answer: {"category": "billing", "reason": "Customer reports a duplicate charge."}

Ticket: My parcel was supposed to arrive last week and the tracking has not moved.
Answer: {"category": "delivery", "reason": "Package is late and tracking is stuck."}

Ticket: The app crashes every time I open the payment page.
Answer: {"category": "technical", "reason": "App crashes, which is a software problem."}

Ticket: Someone used my card to place an order I never made, I did not authorise this.
Answer: {"category": "fraud", "reason": "Customer reports an unauthorised transaction."}"""

# ---------- Checks against the task conditions ----------
for p in (ZERO_SHOT_PROMPT, FEW_SHOT_PROMPT):
    assert "CLASSIFY" in p
    for c in CATEGORIES:
        assert c in p
    assert '{"category": "...", "reason": "..."}' in p

assert "EXAMPLES:" not in ZERO_SHOT_PROMPT       
assert "EXAMPLES:" in FEW_SHOT_PROMPT
assert FEW_SHOT_PROMPT.count("Answer:") >= 2
assert '"category": "fraud"' in FEW_SHOT_PROMPT

print("Both prompts satisfy the conditions.")

Both prompts satisfy the conditions.


## Task 4 - Resilient calls:
Call the mock_llm and from mock_llm import TransientLLMError(which is the temporary-failure error from that file). Function call_with_retry is build with maximum retry(3), exponential backoff(2**n) doubles the wait each time (1, 2, 4), and jitter -- random.uniform(0.9, 1.15) adds a small random change.

### Observation:
The 19 is the total number of temporary errors over all 156 ticket after input guardrail filter. The mock_llm fails on every 9th call, so about 19 different tickets each failed once, and each one succeeded on its first retry. No single ticket used more than 1 of its 3 allowed retries. Total call will be 156(Ticket size) + 19(each failed once and succeeded on its first retry) = 175. And there Zero(0) permanent failure (Exceeds the max_retry) tickets.

## Task 5 - Validate every LLM Answer:
### Observation:
135 classified_json: The LLM reply is parsed as JSON and the category is one of the four allowed values.
(e.g) JSON, as requested: {"category": "billing", "reason": "keyword match"}.

12 classified_keyword: The LLM reply in a sentence instead of keyword, not JSON, so json.loads fails. "re.findall(r"[a-z]+", text.lower())" the split sentences into words, and exactly one allowed category word is found.
(e.g){"category": "The customer have billing-related issue", "reason": "keyword not match"}

9 flagged_invalid_output: They come from two causes:
1. Valid JSON with a category outside the list.
(e.g) {"category": "shipping", "reason": "keyword not match"}.
2. Chatty replies where the keyword rule fails, meaning zero or two-plus allowed category words are found, so the response routed to Human Review.
(e.g){"category": "The customer have no issue", "reason": "keyword not match"}

## Task 6 - Compare zero-shot and few-shot:
### Observation:
    Variable	     What it holds (zero-shot prompt)
    zero_res	  the results table for the zero-shot run
    zero_tokens   total tokens for the zero-shot run
    zero_log	  the retry errors count and retry log lines
    zero_perm	  permanent errors (should be 0)
            zero-shot accuracy 85.7%

    Variable	     What it holds (few-shot prompt)
    few_res	     the results table for the few-shot run
    few_tokens   total tokens for the few-shot run
    few_log	     the retry errors count and retry log lines
    few_perm	 permanent errors (should be 0)
            few-shot accuracy 100%


In [5]:
BASE_DELAY = 0.05
MAX_RETRIES = 3
CATEGORIES = ["billing", "delivery", "technical", "fraud"]

# ---------- Task 4: retry function ----------
def call_with_retry(messages, ticket_id, log):
    for attempt in range(MAX_RETRIES + 1):
        try:
            return mock_llm.chat(messages)
        except TransientLLMError:
            log["errors"] += 1
            if attempt == MAX_RETRIES:
                raise
            wait = BASE_DELAY * 2**attempt * random.uniform(0.9, 1.15)
            log["lines"].append(f"[retry] ticket={ticket_id} retry={attempt} wait={wait:.4f}s")
            time.sleep(wait)


# ---------- Task 5: validation function ----------
def validate(text):
    status, category = "flagged_invalid_output", None
    try:
        data = json.loads(text)
        if isinstance(data, dict) and data.get("category") in CATEGORIES:
            status, category = "classified_json", data["category"]
    except json.JSONDecodeError:
        found = [w for w in re.findall(r"[a-z]+", text.lower()) if w in CATEGORIES]
        if len(found) == 1:
            status, category = "classified_keyword", found[0]
    return status, category


# ---------- One full run (uses both functions) ----------
def run_pipeline(system_prompt):
    mock_llm.reset()
    log = {"errors": 0, "lines": []}
    permanent = 0
    rows, total_tokens = [], 0

    for i, m in enumerate(messages_list):              
        if m is None:
            continue

        tid = df["ticket_id"][i]
        messages = [{"role": "system", "content": system_prompt}] + m

        try:
            response = call_with_retry(messages, tid, log)
        except TransientLLMError:
            permanent += 1
            continue

        total_tokens += response.usage.total_tokens
        status, category = validate(response.choices[0].message.content or "")
        rows.append({"ticket_id": tid, "true": df["true_category"][i],
                     "pred": category, "status": status})

    return pd.DataFrame(rows), total_tokens, log, permanent

# ---------- Task 6: run both prompts ----------
zero_res, zero_tokens, zero_log, zero_perm = run_pipeline(ZERO_SHOT_PROMPT)
few_res, few_tokens, few_log, few_perm = run_pipeline(FEW_SHOT_PROMPT)

# ---------- Task 4 output ----------
print("===== TASK 4 =====")
for line in zero_log["lines"]:
    print(line)
print("tickets answered  :", len(zero_res))
print("permanent errors  :", zero_perm)
print("TransientLLMErrors:", zero_log["errors"])
print("retry log lines   :", len(zero_log["lines"]))

# ---------- Task 5 output ----------
print("\n===== TASK 5 =====")
print(zero_res["status"].value_counts())
print("Ticket IDs routed for human review:", zero_res[zero_res["status"] == "flagged_invalid_output"]["ticket_id"].tolist())

# ---------- Task 6 output ----------
print("\n===== TASK 6 =====")
for name, res, tok in [("Zero-shot", zero_res, zero_tokens), ("Few-shot", few_res, few_tokens)]:
    cls = res[res["status"].isin(["classified_json", "classified_keyword"])]
    n = len(cls)
    ok = int((cls["true"] == cls["pred"]).sum())
    print(f"{name}: classified={n}, accuracy={ok/n*100:.1f}% ({ok} of {n}), "
          f"total tokens={tok}, tokens per classified={tok/n:.1f}")

zero_cls = zero_res[zero_res["status"].isin(["classified_json", "classified_keyword"])]
print("\nZero-shot confusion table (rows = true, columns = predicted):")
print(pd.crosstab(zero_cls["true"], zero_cls["pred"]))

===== TASK 4 =====
[retry] ticket=T0002 retry=0 wait=0.0506s
[retry] ticket=T0116 retry=0 wait=0.0472s
[retry] ticket=T0153 retry=0 wait=0.0571s
[retry] ticket=T0039 retry=0 wait=0.0512s
[retry] ticket=T0152 retry=0 wait=0.0466s
[retry] ticket=T0043 retry=0 wait=0.0484s
[retry] ticket=T0111 retry=0 wait=0.0567s
[retry] ticket=T0155 retry=0 wait=0.0462s
[retry] ticket=T0100 retry=0 wait=0.0553s
[retry] ticket=T0068 retry=0 wait=0.0501s
[retry] ticket=T0135 retry=0 wait=0.0556s
[retry] ticket=T0009 retry=0 wait=0.0553s
[retry] ticket=T0067 retry=0 wait=0.0466s
[retry] ticket=T0106 retry=0 wait=0.0561s
[retry] ticket=T0097 retry=0 wait=0.0533s
[retry] ticket=T0003 retry=0 wait=0.0519s
[retry] ticket=T0019 retry=0 wait=0.0517s
[retry] ticket=T0086 retry=0 wait=0.0563s
[retry] ticket=T0065 retry=0 wait=0.0473s
tickets answered  : 156
permanent errors  : 0
TransientLLMErrors: 19
retry log lines   : 19

===== TASK 5 =====
status
classified_json           135
classified_keyword         12
flag

## Task 7 - Budget, Routing and Output:

In [6]:
os.makedirs("outputs", exist_ok=True)

# customer segments from Part B (missing customers are treated as not premium)
seg = pd.read_csv("outputs/customer_segments.csv", keep_default_na=False, dtype=str)
seg = seg.set_index("customer_id")

def route_ticket(status, category, is_premium):
    if status in ("skipped_empty", "flagged_long", "flagged_invalid_output", "budget_exhausted"):
        return "human_review", "NORMAL"
    if category == "fraud":
        return "security_team", "HIGH"
    if is_premium:
        return "premium_desk", "HIGH"
    return "support_queue", "NORMAL"


def run_with_budget(system_prompt, token_budget):
    mock_llm.reset()                                    # right before each full run
    log = {"errors": 0, "lines": []}
    running_total = 0
    rows = []

    for i, m in enumerate(messages_list):               # all 164 tickets, file order
        tid = df["ticket_id"][i]
        cid = str(df["customer_id"][i])
        category, tokens_used = "", 0

        if m is None:                                   # input guardrails (Task 1)
            status = "skipped_empty" if str(df["text"][i]).strip() == "" else "flagged_long"

        elif running_total >= token_budget:             # budget check BEFORE the call
            status = "budget_exhausted"

        else:
            messages = [{"role": "system", "content": system_prompt}] + m
            try:
                response = call_with_retry(messages, tid, log)
                tokens_used = response.usage.total_tokens
                running_total += tokens_used
                status, cat = validate(response.choices[0].message.content or "")
                category = cat or ""
            except TransientLLMError:
                status = "flagged_invalid_output"       # permanent error -> human review

        # customer segment
        if cid in seg.index:
            segment_name = seg.loc[cid, "segment_name"]
            is_premium = str(seg.loc[cid, "is_premium"]).strip().lower() in ("true", "1", "yes")
        else:
            segment_name, is_premium = "", False

        route, priority = route_ticket(status, category, is_premium)

        rows.append({"ticket_id": tid, "customer_id": cid, "status": status,
                     "category": category, "segment_name": segment_name,
                     "priority": priority, "route": route, "tokens_used": tokens_used})

    return pd.DataFrame(rows)


# ---------- (a) budget demo: zero-shot, TOKEN_BUDGET = 6000 ----------
TOKEN_BUDGET = 6000
demo = run_with_budget(ZERO_SHOT_PROMPT, TOKEN_BUDGET)
demo.to_csv("outputs/triage_budget_demo.csv", index=False)

n_exhausted = (demo["status"] == "budget_exhausted").sum()
print("budget used in ZERO_SHOT           :", TOKEN_BUDGET)
print("budget_exhausted count in ZERO_SHOT:", n_exhausted)      # must be at least 1
print("tokens used in demo in ZERO_SHOT   :", demo["tokens_used"].sum())

# ---------- (b) final output: few-shot, TOKEN_BUDGET = 100000 ----------
final = run_with_budget(FEW_SHOT_PROMPT, 100000)
final.to_csv("outputs/triage_results.csv", index=False)

print("tokens used in demo in FEW_SHOT   :", final["tokens_used"].sum())
print("\nFinal status counts for FEW_SHOT:")
print(final["status"].value_counts())
print("\nRoute counts for ZERO_SHOT:")
print(demo["route"].value_counts())
print("\nPriority counts for ZERO_SHOT:")
print(demo["priority"].value_counts())
print("\nrows saved in ZERO_SHOT:", len(demo))

budget used in ZERO_SHOT           : 6000
budget_exhausted count in ZERO_SHOT: 106
tokens used in demo in ZERO_SHOT   : 6065
tokens used in demo in FEW_SHOT   : 38728

Final status counts for FEW_SHOT:
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
skipped_empty               4
flagged_long                4
Name: count, dtype: int64

Route counts for ZERO_SHOT:
route
human_review     116
support_queue     38
security_team      6
premium_desk       4
Name: count, dtype: int64

Priority counts for ZERO_SHOT:
priority
NORMAL    154
HIGH       10
Name: count, dtype: int64

rows saved in ZERO_SHOT: 164


In [7]:
few_cls = few_res[few_res["status"].isin(["classified_json", "classified_keyword"])]
few_per_ticket = few_tokens / len(few_cls)
print("Average tokens per ticket in FEW_SHOT:", round(few_per_ticket, 1))
print("Total number of tokens used in FEW_SHOT:", few_tokens)

zero_cls = zero_res[zero_res["status"].isin(["classified_json", "classified_keyword"])]
zero_per_ticket = zero_tokens / len(zero_cls)
print("Total number of tokens used in ZERO_SHOT:", zero_tokens)
print("Average tokens per ticket in ZERO_SHOT:", round(zero_per_ticket, 1))

print("Difference extra tokens per ticket:", round(few_per_ticket - zero_per_ticket, 1))

Average tokens per ticket in FEW_SHOT: 263.5
Total number of tokens used in FEW_SHOT: 38728
Total number of tokens used in ZERO_SHOT: 18916
Average tokens per ticket in ZERO_SHOT: 128.7
Difference extra tokens per ticket: 134.8


## Observation:
1. The demo run (zero-shot, budget 6000). Report the budget used and the budget_exhausted count (106). Saved in outputs/triage_budget_demo.csv.

2. The final run (few-shot, budget 100000). Report the status, route and priority counts (human_review as 17, budget_exhausted as 0 and so on). Saved in outputs/triage_results.csv.